In [223]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [224]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [225]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

In [226]:
def get_exercise_type(row):
    tipos = [row.tipo_ejercicio_x, row.tipo_ejercicio_y]
    tipo  = [i for i in tipos if pd.notna(i)]
    if len(tipo) > 0:
        return tipo[0]
    else:
        return pd.NA

## Combinar enunciados con clasificaciones

Cargar enunciados

In [227]:
df_enun = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding="utf-8")

Establecer `id` como columna índice, para poder hacer `update`

In [228]:
df_enun = df_enun.set_index("id")

In [229]:
df_enun.head(2)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen
id,,,,,,,,,,
fisica_campo_electrico_y_magnetico_2025_julio_ejercicio_a,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,NaN,Discuta la veracidad de las siguientes afirmac...,FISICA. 2025. JULIO. EJERCICIO a,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
fisica_campo_electrico_y_magnetico_2025_julio_ejercicio_b1,Física,2025,Julio,Ejercicio B1,campo eléctrico y magnético,NaN,Dos conductores rectilíneos muy largos se disp...,FISICA. 2025. JULIO. EJERCICIO b1,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...


In [233]:
df_enun.columns

Index(['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
       'tipo_ejercicio', 'enunciado', 'info_ejercicio', 'archivo', 'imagen'],
      dtype='str')

In [232]:
df_enun[(df_enun.asignatura == "Física") & (~df_enun.tipo_ejercicio.isna())].tipo_ejercicio

id
fisica_campo_gravitatorio_2026_junio_ejercicio_a                   verdadero / falso
fisica_campo_gravitatorio_2026_junio_ejercicio_b                     plano_inclinado
fisica_campo_gravitatorio_2025_julio_ejercicio_a                   verdadero / falso
fisica_campo_gravitatorio_2025_julio_ejercicio_b1                 distribución_masas
fisica_campo_gravitatorio_2025_julio_ejercicio_b2                          satélites
                                                                         ...        
fisica_campo_gravitatorio_2016_reserva_4_ejercicio_1_opcion_a     distribución_masas
fisica_campo_gravitatorio_2016_reserva_4_ejercicio_3_opcion_b        plano_inclinado
fisica_campo_gravitatorio_2016_septiembre_ejercicio_1_opcion_b     verdadero / falso
fisica_campo_gravitatorio_2016_septiembre_ejercicio_3_opcion_a             satélites
fisica_campo_gravitatorio_2016_septiembre_ejercicio_3_opcion_a             satélites
Name: tipo_ejercicio, Length: 122, dtype: str

In [186]:
df_enun.columns

Index(['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
       'tipo_ejercicio', 'enunciado', 'info_ejercicio', 'archivo', 'imagen'],
      dtype='str')

## Cargar ejercicios clasificados **a incorporar**

In [192]:
df_classified = pd.read_csv(root / "csv/ejercicios_clasificados_matrices_ccss.csv")

In [193]:
df_classified.head()

,tipo_ejercicio,confianza,id
0,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2001_junio...
1,valores tenga inversa,alta,mates_ccss_matrices_y_determinantes_2001_reser...
2,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2001_reser...
3,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2001_reser...
4,ecuación matricial,alta,mates_ccss_matrices_y_determinantes_2002_reser...


Establecer `id` como columna índice, para poder hacer `update`

In [194]:
df_classified = df_classified.set_index("id")

In [195]:
df_classified.head(2)

,tipo_ejercicio,confianza
id,,
mates_ccss_matrices_y_determinantes_2001_junio_ejercicio_1_opcion_b,ecuación matricial,alta
mates_ccss_matrices_y_determinantes_2001_reserva_1_ejercicio_1_opcion_a,valores tenga inversa,alta


In [196]:
df_classified.loc[:, ["asignatura", "tema"]] = ["Mates CCSS", "matrices y determinantes"]

In [197]:
df_classified[df_classified.tipo_ejercicio.isna()]

,tipo_ejercicio,confianza,asignatura,tema
id,,,,


Guardar cambios en el archivo (previo `reset_index()`)

In [200]:
df_classified.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados_matrices_ccss.csv", index = False, encoding="utf-8")

### Inspeccionar clasificados

In [201]:
df_classified.columns

Index(['tipo_ejercicio', 'confianza', 'asignatura', 'tema'], dtype='str')

In [202]:
df_classified.sample()

,tipo_ejercicio,confianza,asignatura,tema
id,,,,
mates_ii_matrices_y_determinantes_2002_junio_ejercicio_3_opcion_a,ecuación matricial,media,Mates CCSS,matrices y determinantes


Actualizar los ejercicios clasificados y resetear índice

In [203]:
df_enun.update(df_classified.tipo_ejercicio)
df_enun = df_enun.reset_index(drop = False)

In [204]:
rearranged_cols = ['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
                   'tipo_ejercicio','enunciado',
                   "id",
                   'info_ejercicio', 'archivo', 'imagen'
                   ]
df_enun = df_enun[rearranged_cols]

Exportar enunciados clasificados actualizado

In [207]:
df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
               index = False, encoding="utf-8")

Opcional: insertar enunciados limpios de matrices

In [208]:
df_clean_st = pd.read_csv(root / "classifier/matrices_limpio.csv")

In [209]:
df_clean_st.sample()

,id,enunciado
250,mates_ii_matrices_y_determinantes_2018_reserva...,Considera la matriz M = [matriz]. Sabiendo que...


In [212]:
df_enun = df_enun.set_index("id")
df_enun.update(df_clean_st.set_index("id").enunciado)

In [215]:
df_enun[df_enun.tema == "matrices y determinantes"].sample(3)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen
id,,,,,,,,,,
mates_ii_matrices_y_determinantes_2012_reserva_1_ejercicio_3_opcion_a,Mates II,2012,Reserva 1,"Ejercicio 3, opción A",matrices y determinantes,ecuación matricial,C onsidera las matrices: [matriz] A= [matriz] ...,MATEMÁTICAS II. 2012. RESERVA 1. EJERCICIO 3. ...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...
mates_ii_matrices_y_determinantes_2019_reserva_1_ejercicio_3_opcion_a,Mates II,2019,Reserva 1,"Ejercicio 3, opción A",matrices y determinantes,valores tenga inversa|ecuación matricial,Dadas las matrices A= m-1 m 0 y B = [matriz]\n...,MATEMÁTICAS II. 2019. RESERVA 1. EJERCICIO 3. ...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...
mates_ccss_matrices_y_determinantes_2006_reserva_2_ejercicio_1_opcion_a,Mates CCSS,2006,Reserva 2,"Ejercicio 1, opción A",matrices y determinantes,ecuación matricial,Sean las matricesA= [matriz] y B= [matriz]. Ca...,SOCIALES II. 2006 RESERVA 2. EJERCICIO 1. OPCI...,source_pdf_files/Mates CCSS/matrices_determina...,enunciados_ejercicios/Mates CCSS/matrices_dete...


Exportar enunciados clasificados actualizado

In [216]:
df_enun.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados.csv",
                                         index = False, encoding="utf-8")